# 03 — Augmentation Study

**Question**: which augmentation pipeline produces the most robust classifier against AI-generated images?
And more interestingly: which perturbations hurt the model most?

**Setup**: same ResNet18, same seed, same split. Only `augmentation_name` changes across runs.

**Augmentations tested**:
| Name | What it does |
|---|---|
| `none` | No augmentation — pure baseline |
| `weak` | Random crop + h-flip (← notebook 02 baseline) |
| `jpeg_compression` | JPEG encode/decode quality=30 — destroys high-freq artefacts |
| `gaussian_blur` | Blur σ in [0.1, 2.0] — smooths texture |
| `gaussian_noise` | Additive noise std=0.05 — perturbs pixel values |
| `color_jitter` | Brightness/contrast/saturation/hue jitter |
| `random_crop` | Aggressive crop (scale 0.6–1.0) — tests spatial robustness |

**Hypothesis**: if `jpeg_compression` or `gaussian_blur` cause the biggest accuracy drop,
the model relies on high-frequency texture artefacts typical of GAN/diffusion generators.

Figures → `reports/figures/03_augmentation_study/`  
CSV    → `reports/03_augmentation_results.csv`

## 0. Setup

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT / 'datasets').exists()
):
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Cannot locate project root')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import torch

from src.train_supervised import train_supervised
from src.augmentations import list_augmentation_names
from src.utils import (
    fig_path, save_figure, save_results_csv,
    plot_roc_curve, plot_confusion_matrix,
)

REPORTS_DIR = str(PROJECT_ROOT / 'reports')
NB          = '03_augmentation_study'

%matplotlib inline

print('Augmentations available:', list_augmentation_names())

## 1. Shared config

In [ ]:
BASE_CONFIG = {
    'data_root':      str(PROJECT_ROOT / 'datasets' / 'raw'),
    'model_name':     'resnet18',
    'num_epochs':     8,
    'batch_size':     32,
    'image_size':     224,
    'num_workers':    4,
    'split_ratios':   (0.7, 0.15, 0.15),
    'seed':           42,
    'checkpoint_dir': str(PROJECT_ROOT / 'checkpoints'),
    'device':         'cuda' if torch.cuda.is_available() else 'cpu',
}

AUG_NAMES = list_augmentation_names()  # all supervised pipelines
print(f'Will train {len(AUG_NAMES)} models:', AUG_NAMES)

## 2. Run all augmentation experiments

In [ ]:
all_results = {}  # aug_name -> results dict

for aug_name in AUG_NAMES:
    print(f'\n{"-"*60}')
    print(f'Training with augmentation: {aug_name}')
    print(f'{"-"*60}')
    cfg = {**BASE_CONFIG, 'augmentation_name': aug_name}
    res = train_supervised(cfg)
    all_results[aug_name] = res
    print(f'  test_acc={res["test_accuracy"]:.4f}  F1={res["test_f1"]:.4f}  AUC={res["test_auc"]:.4f}  MCC={res["test_mcc"]:.4f}')

print('\nAll runs complete.')

## 3. Results table

In [ ]:
rows = []
for aug_name, res in all_results.items():
    m = res['test_metrics_full']
    rows.append({
        'augmentation':      aug_name,
        'accuracy':          round(m['accuracy'],          4),
        'balanced_accuracy': round(m['balanced_accuracy'], 4),
        'f1':                round(m['f1'],                4),
        'auc':               round(m['auc'],               4),
        'avg_precision':     round(m['avg_precision'],     4),
        'mcc':               round(m['mcc'],               4),
        'precision':         round(m['precision'],         4),
        'recall':            round(m['recall'],            4),
        'specificity':       round(m['specificity'],       4),
    })

df = pd.DataFrame(rows).sort_values('auc', ascending=False).reset_index(drop=True)
display(df.style
    .background_gradient(subset=['auc', 'f1', 'mcc'], cmap='RdYlGn')
    .format({c: '{:.4f}' for c in df.columns if c != 'augmentation'})
)

## 4. Bar chart — key metrics per augmentation

In [ ]:
metrics_to_plot = ['accuracy', 'f1', 'auc', 'mcc']
aug_labels = df['augmentation'].tolist()
x = np.arange(len(aug_labels))
n_metrics = len(metrics_to_plot)
width = 0.18
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

fig, ax = plt.subplots(figsize=(13, 5))
for i, (metric, color) in enumerate(zip(metrics_to_plot, colors)):
    vals = df[metric].tolist()
    offset = (i - n_metrics / 2 + 0.5) * width
    bars = ax.bar(x + offset, vals, width, label=metric, color=color, alpha=0.85)

# highlight weak (baseline) with a vertical dashed line
if 'weak' in aug_labels:
    bi = aug_labels.index('weak')
    ax.axvline(bi, color='gray', linestyle='--', linewidth=1.2, alpha=0.6, label='weak (baseline)')

ax.set_xticks(x)
ax.set_xticklabels(aug_labels, rotation=20, ha='right', fontsize=9)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Augmentation study — test set metrics (sorted by AUC)')
ax.legend(loc='lower right', fontsize=9)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
save_figure(fig, fig_path(REPORTS_DIR, NB, 'metrics_bar.png'), show=True)

## 5. AUC drop vs baseline (signal erosion chart)

In [ ]:
# This chart answers: 'which augmentation hurts the model most?'
# A large negative delta means the model was relying on a signal that aug destroyed.
baseline_auc = all_results['weak']['test_metrics_full']['auc']

delta_rows = []
for aug_name, res in all_results.items():
    delta_rows.append({
        'augmentation': aug_name,
        'auc':          res['test_metrics_full']['auc'],
        'delta_auc':    res['test_metrics_full']['auc'] - baseline_auc,
        'f1':           res['test_metrics_full']['f1'],
        'delta_f1':     res['test_metrics_full']['f1'] - all_results['weak']['test_metrics_full']['f1'],
    })
df_delta = pd.DataFrame(delta_rows).sort_values('delta_auc').reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, title in zip(axes, ['delta_auc', 'delta_f1'], ['Δ AUC vs weak', 'Δ F1 vs weak']):
    clrs = ['tomato' if v < 0 else 'steelblue' for v in df_delta[col]]
    ax.barh(df_delta['augmentation'], df_delta[col], color=clrs)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel('Delta (positive = better than weak)')
    ax.grid(axis='x', alpha=0.3)
fig.suptitle('Signal erosion: how much each augmentation degrades performance', fontsize=11)
fig.tight_layout()
save_figure(fig, fig_path(REPORTS_DIR, NB, 'signal_erosion.png'), show=True)

## 6. ROC curves — all augmentations overlaid

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='random')

cmap = matplotlib.colormaps.get_cmap('tab10').resampled(len(all_results))
for i, (aug_name, res) in enumerate(all_results.items()):
    y_true = res['test_y_true']
    y_prob = res['test_y_prob']
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    lw = 2.5 if aug_name == 'weak' else 1.2
    ax.plot(fpr, tpr, color=cmap(i), linewidth=lw,
            label=f'{aug_name} (AUC={auc:.3f})')

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC curves — all augmentations')
ax.legend(fontsize=8, loc='lower right')
fig.tight_layout()
save_figure(fig, fig_path(REPORTS_DIR, NB, 'roc_overlay.png'), show=True)

## 7. Confusion matrices — most vs least robust augmentation

In [ ]:
best_aug  = df.iloc[0]['augmentation']   # highest AUC
worst_aug = df.iloc[-1]['augmentation']  # lowest AUC
print(f'Best : {best_aug}  (AUC={df.iloc[0]["auc"]:.4f})')
print(f'Worst: {worst_aug}  (AUC={df.iloc[-1]["auc"]:.4f})')

for aug_name, label in [(best_aug, 'best'), (worst_aug, 'worst')]:
    res = all_results[aug_name]
    plot_confusion_matrix(
        res['test_y_true'], res['test_y_pred'],
        title=f'Confusion Matrix — {aug_name} ({label})',
        path=fig_path(REPORTS_DIR, NB, f'confusion_matrix_{label}.png'),
        show=True,
    )

## 8. Training curves comparison (val accuracy)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
cmap = matplotlib.colormaps.get_cmap('tab10').resampled(len(all_results))

for i, (aug_name, res) in enumerate(all_results.items()):
    epochs    = [h['epoch']        for h in res['history']]
    val_acc   = [h['val_accuracy']  for h in res['history']]
    train_loss= [h['train_loss']    for h in res['history']]
    lw = 2.5 if aug_name == 'weak' else 1.2
    axes[0].plot(epochs, val_acc,    color=cmap(i), linewidth=lw, label=aug_name, marker='o', markersize=3)
    axes[1].plot(epochs, train_loss, color=cmap(i), linewidth=lw, label=aug_name, marker='o', markersize=3)

axes[0].set_title('Val Accuracy per epoch'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[1].set_title('Train Loss per epoch');   axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
for ax in axes:
    ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)
fig.suptitle('Training dynamics — all augmentations', fontsize=11)
fig.tight_layout()
save_figure(fig, fig_path(REPORTS_DIR, NB, 'training_curves_all.png'), show=True)

## 9. Save full results to CSV

In [ ]:
save_results_csv(rows, str(Path(REPORTS_DIR) / '03_augmentation_results.csv'))

# also save per-epoch history for all runs
all_history = []
for aug_name, res in all_results.items():
    for h in res['history']:
        all_history.append({'augmentation': aug_name, **h})
save_results_csv(all_history, str(Path(REPORTS_DIR) / '03_augmentation_history.csv'))

print(f'Done. Figures → reports/figures/{NB}/')
print( 'CSV   → reports/03_augmentation_results.csv')
print( 'CSV   → reports/03_augmentation_history.csv')